# 1. Lọc Bình Dương từ dữ liệu gốc

In [3]:
import duckdb

# Tên file gốc theo ảnh là vietnam_real_estate.csv, hãy đối chiếu lại trước khi chạy
RAW_SOURCE = '../datas/raw/vietnam_real_estate.csv'
RAW_BINH_DUONG = '../datas/raw/binh_duong_real_estates.csv'

duckdb.query(f"""
    COPY (
        SELECT *
        FROM '{RAW_SOURCE}'
        WHERE lower(province_name) LIKE '%bình dương%'
           OR lower(province_name) LIKE '%binh duong%'
    ) TO '{RAW_BINH_DUONG}' (HEADER, DELIMITER ',')
""")

# Đọc dữ liệu đã lọc

In [4]:
import pandas as pd

df = pd.read_csv('../datas/raw/binh_duong_real_estates.csv', encoding='utf-8-sig')
total_rows = len(df)
print("Số dòng:", f"{total_rows:,}", "| Số cột:", df.shape[1])

Số dòng: 42,783 | Số cột: 19


# Dữ liệu thiếu

In [5]:
missing = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Rate (%)': (df.isnull().sum() / total_rows * 100).round(2)
}).sort_values('Missing_Count', ascending=False)

print(missing[missing['Missing_Count'] > 0].to_string())

                   Missing_Count  Missing_Rate (%)
house_depth                41984             98.13
floor_count                37184             86.91
road_width                 35255             82.40
balcony_direction          34536             80.72
house_direction            27772             64.91
frontage_width             23811             55.66
project_name               23004             53.77
street_name                22874             53.47
bathroom_count             19268             45.04
bedroom_count              18285             42.74
ward_name                   3375              7.89
price                       3324              7.77


# Trùng lặp

In [6]:
BIZ_COLS = ['name', 'price', 'area', 'ward_name', 'district_name']

exact_dups = df.duplicated().sum()
biz_dups = df.duplicated(subset=BIZ_COLS, keep='first').sum()

print(f"Trùng tuyệt đối: {exact_dups:,} ({exact_dups/total_rows*100:.4f}%)")
print(f"Trùng nghiệp vụ: {biz_dups:,} ({biz_dups/total_rows*100:.2f}%)")

Trùng tuyệt đối: 1 (0.0023%)
Trùng nghiệp vụ: 3,355 (7.84%)


# Giá trị không hợp lệ và định dạng

In [7]:
price = pd.to_numeric(df['price'], errors='coerce')
area = pd.to_numeric(df['area'], errors='coerce')

print("price <= 0:", (price <= 0).sum())
print("area < 10 m2:", (area < 10).sum())

for col in ['name', 'district_name', 'ward_name', 'street_name', 'project_name']:
    s = df[col].astype('string')
    print(f"{col} có khoảng trắng thừa:", (s != s.str.strip()).sum())

print("\nCác giá trị hướng nhà:")
print(df['house_direction'].value_counts(dropna=False).to_string())

price <= 0: 0
area < 10 m2: 2
name có khoảng trắng thừa: 0
district_name có khoảng trắng thừa: 0
ward_name có khoảng trắng thừa: 0
street_name có khoảng trắng thừa: 0
project_name có khoảng trắng thừa: 129

Các giá trị hướng nhà:
house_direction
NaN           27772
Nam            1942
Đông Nam       1802
Đông           1440
Bắc            1418
Đông Bắc       1320
Đông - Nam     1314
Tây Bắc        1200
Tây            1103
Tây - Bắc       973
Tây Nam         921
Đông - Bắc      873
Tây - Nam       705


# Kiểm tra ngày đăng

In [8]:
dt_iso = pd.to_datetime(df['published_at'], format='ISO8601', errors='coerce')
dt_dayfirst = pd.to_datetime(df['published_at'], format='mixed', dayfirst=True, errors='coerce')

comparison = pd.DataFrame({
    'Chuẩn ISO': dt_iso.dt.month.value_counts(),
    'Khai báo dayfirst=True': dt_dayfirst.dt.month.value_counts()
}).sort_index()

print(comparison.fillna(0).astype(int))
print("Không đọc được:", dt_iso.isna().sum())
print("Khoảng thời gian:", dt_iso.min(), "đến", dt_iso.max())

              Chuẩn ISO  Khai báo dayfirst=True
published_at                                   
1                     0                    1232
2                     0                    1187
3                     0                    1453
4                     0                    1482
5                     0                    1198
6                  4577                    4434
7                  6615                    5056
8                  6315                    5451
9                  5942                    5039
10                 6586                    5455
11                 5769                    5033
12                 6979                    5763
Không đọc được: 0
Khoảng thời gian: 2025-06-01 01:41:55.422000 đến 2025-12-30 22:49:43.622000


# e. Ngoại lệ đơn giá và tổng giá

In [9]:
valid = df[(price > 0) & (area >= 10)].copy()
ppm = price[valid.index] / area[valid.index]

print("Số dòng trong tập valid:", len(valid))
print("Đơn giá min/max:", f"{ppm.min():,.0f}", "/", f"{ppm.max():,.0f}")

q1, q3 = ppm.quantile([0.25, 0.75])
iqr = q3 - q1
iqr_out = ((ppm < q1 - 1.5*iqr) | (ppm > q3 + 1.5*iqr)).sum()
print("Ngoại lệ theo IQR (chỉ để chẩn đoán):", iqr_out)
print(f"Tỷ lệ ngoại lệ IQR: {iqr_out / len(valid) * 100:.2f}%")
print("Tổng giá trên 500 tỷ:", (price > 5e11).sum())

Số dòng trong tập valid: 39457
Đơn giá min/max: 15 / 54,385,964,912
Ngoại lệ theo IQR (chỉ để chẩn đoán): 1534
Tỷ lệ ngoại lệ IQR: 3.89%
Tổng giá trên 500 tỷ: 19
